# Roadmap

1. Primero probar con una grid search con tamaño de ventana predefinido sin seleccion de caracteristicas.

2. Probar distintos tamaños de ventanas, usando la configuración obtenida anteriormente mediante grid search.

3. Selección de características teniendo en cuenta el tamaño de ventana del paso anterior y la configuración de hiperparametros encontrada.

4. Evaluacion de modelos y ajuste de hiperparametros.

# Funciones:

In [6]:
import os
import pandas as pd
import numpy as np
import neurokit2 as nk
import ast
from sklearn.model_selection import GroupShuffleSplit, GridSearchCV, GroupKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

import seaborn as sns
import matplotlib.pyplot as plt

## Extracción de ventanas con solapamiento:

In [2]:

def extract_windows(path, sr=500, window_size = 30, overlap_sec = 10):

    SAMPLES_PER_WINDOW = sr * window_size
    OVERLAP_SAMPLES = sr * overlap_sec
    STEP_SIZE = SAMPLES_PER_WINDOW - OVERLAP_SAMPLES

    all_session_rows = []

    for subject in os.listdir(path):
        subject_path = os.path.join(path, subject)
        
        # skip non directories
        if not os.path.isdir(subject_path):
            continue
            
        for file in os.listdir(subject_path):
            file_path = os.path.join(subject_path, file)
            
            # Load the data
            df_physio = pd.read_csv(file_path)
            
            # Extract activity from filename 
            activity = os.path.basename(file_path).split('.')[0].split('_')[1]

            # Extract the signals
            raw_signals = df_physio[['ECG', 'EDA', 'RR']].values
            total_samples = len(raw_signals)
            
            # If the file is shorter than one window, skip it to avoid errors
            if total_samples < SAMPLES_PER_WINDOW:
                continue

            # how many overlapping windows we can fit
            num_windows = (total_samples - SAMPLES_PER_WINDOW) // STEP_SIZE + 1

            # Slide the window across the data
            for w in range(num_windows):
                start_idx = w * STEP_SIZE
                end_idx = start_idx + SAMPLES_PER_WINDOW
                
                # Slice the array
                window_data = raw_signals[start_idx:end_idx]

                all_session_rows.append({
                    'subject': subject,
                    'activity': activity,
                    'window_id': w,
                    'ECG': window_data[:, 0].tolist(), 
                    'EDA': window_data[:, 1].tolist(),
                    'RR':  window_data[:, 2].tolist()
                })

    return pd.DataFrame(all_session_rows)

## Extracción de características con Neurokit2 y preprocesamiento:

In [7]:
# extraction function for a single row
def extract_nk2_features(row, sr=500):
    ecg_raw = np.array(row['ECG'])
    eda_raw = np.array(row['EDA'])
    rsp_raw = np.array(row['RR'])
    
    features = {}
    
    # ECG
    try:
        # clean and process the raw signal
        ecg_signals, _ = nk.ecg_process(ecg_raw, sampling_rate=sr)
        ecg_feat = nk.ecg_intervalrelated(ecg_signals) #features
        for col in ecg_feat.columns:
            features[col] = ecg_feat[col].iloc[0]
    except Exception:
        pass # If the window is too noisy, leave features empty (NaN)

    # EDA
    try:
        eda_signals, _ = nk.eda_process(eda_raw, sampling_rate=sr)
        eda_feat = nk.eda_intervalrelated(eda_signals)
        for col in eda_feat.columns:
            features[col] = eda_feat[col].iloc[0]
    except Exception:
        pass

    # RR
    try:
        rsp_signals, _ = nk.rsp_process(rsp_raw, sampling_rate=sr)
        rsp_feat = nk.rsp_intervalrelated(rsp_signals)
        for col in rsp_feat.columns:
            features[col] = rsp_feat[col].iloc[0]
    except Exception:
        pass
        
    # Return as pd.Series
    return pd.Series(features)


#The [[v]] values are obviously wrong, so we need to extract them
def extract_scalar(val):
    if isinstance(val, str):
        try:
            val = ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return np.nan

    try:
        return float(val[0][0])
    except (TypeError, IndexError, ValueError):
        return np.nan
    


def preprocess(df, threshold=30):
    '''
    Preprocesamiento centrado en la limpieza del dataframe de features.
    1. Elimina infs.
    2. Elimina columnas con NaNs superiores al porcentaje igual a "threshold"
    3. Elimina filas con algún NaN.
    4. Extrae los valores con formato [[v]].
    '''
    df = df.replace([np.inf, -np.inf], np.nan)

    missing_percentages = df.isna().sum() / len(df) * 100

    cols_to_drop = missing_percentages[missing_percentages > threshold].index
    df = df.drop(columns=cols_to_drop)
    df = df.dropna()

    
    feature_cols = [col for col in df.select_dtypes(include="object").columns if col not in ['subject', 'activity', 'window_id']]
    for col in feature_cols:
        df[col] = df[col].apply(extract_scalar)
    
    return df

def normalize(df, by=['Baseline', 'Relax'], clean = True):
    '''
    Normalización por sujeto usando Z-score, en base a la actividad especificada con param "by".
    '''

    feature_cols = [col for col in df.columns if col not in ['subject', 'activity', 'window_id']]

    # Aislar los datos de baseline para calcular la referencia
    #df_baseline = df[df['activity'] == 'Baseline']
    df_baseline = df[df['activity'].isin(by)]


    # Calcular la media y std de cada característica durante el baseline, agrupado por sujeto
    baseline_means = df_baseline.groupby('subject')[feature_cols].mean().reset_index()
    baseline_stds = df_baseline.groupby('subject')[feature_cols].std().reset_index()

    # Fusionar (merge) estas medias con el dataframe original
    # con sufijos a las columnas para diferenciarlas
    df_norm = pd.merge(df, baseline_means, on='subject', suffixes=('', '_mean'))
    df_norm = pd.merge(df_norm, baseline_stds, on='subject', suffixes=('', '_std'))

    # Aplicar la normalización z score
    # Restamos el valor base al valor de la ventana actual para aislar el cambio
    for col in feature_cols:
        df_norm[col] = (df_norm[col] - df_norm[f"{col}_mean"]) / (df_norm[f"{col}_std"] + 1e-8)

    # Eliminar las columnas _mean, _std
    columnas_auxiliares = [f"{col}_mean" for col in feature_cols] + [f"{col}_std" for col in feature_cols]
    df_norm.drop(columns=columnas_auxiliares, inplace=True)

    if clean:
        df_norm = preprocess(df_norm)

    return df_norm


def feature_extraction_and_normalization(df, threshold=30, norm_by=['Baseline', 'Relax'], clean = True):
    '''
    Aplicar el proceso global de extracción de características + limpieza + normalización + limpieza
    '''
    features_df = df.apply(extract_nk2_features, axis=1) #apply feature extraction per row
    df_features = pd.concat(
    [df[['subject', 'activity', 'window_id']], features_df], 
    axis=1
    )

    df_features = preprocess(df_features, threshold=threshold)
    df_features = normalize(df_features, by = norm_by, clean = clean)

    return df_features

## Split de datos por sujeto y evaluación de modelos:

In [4]:

RANDOM_STATE = 123

def get_group_idx(df, test_size=0.20, stress_act = 'Counting2', non_stress_act = 'Breathing'):
    df_model = df[df['activity'].isin([stress_act, non_stress_act])].copy()

    # Creamos la etiqueta: 0 No estrés, 1 Estrés
    df_model['target'] = (df_model['activity'] == stress_act).astype(int)

    # Separamos las características (X) de las etiquetas (y) y los grupos (Sujetos)
    feature_cols = [col for col in df_model.columns if col not in ['subject', 'activity', 'window_id', 'target']]

    X = df_model[feature_cols]
    y = df_model['target']
    subjects = df_model['subject']


    # Configuramos la división: 80% sujetos para Train, 20% sujetos para test
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=RANDOM_STATE)

    # Obtenemos los índices de las filas para cada conjunto
    # como n_splits = 1, pues haciendo next, obtenemos el unico split realizado...
    train_idx, test_idx = next(gss.split(X, y, groups=subjects))

    return train_idx, test_idx, X, y, subjects


def train_test_split_per_subject(df, test_size=0.20, stress_act = 'Counting2', non_stress_act = 'Breathing'):
    train_idx, test_idx, X, y, subjects = get_group_idx(df, test_size=test_size, stress_act = stress_act, non_stress_act = non_stress_act)

    # Creamos los DataFrames finales de entrenamiento y prueba
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_test = X.iloc[test_idx]
    y_test = y.iloc[test_idx]

    print(f"Sujetos en Train: {subjects.iloc[train_idx].nunique()}")
    print(f"Sujetos en Test: {subjects.iloc[test_idx].nunique()}")

    return X_train, X_test, y_train, y_test

def evaluate_and_plot(y_test, y_pred, stress_act = 'Counting2', non_stress_act = 'Breathing'):

    print(classification_report(y_test, y_pred, target_names=[f'{non_stress_act}(0)', f'{stress_act}(1)']))

    # Matriz de confusión
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Predicción No Estrés', 'Predicción Estrés'],
                yticklabels=['Realidad No Estrés', 'Realidad Estrés'])
    plt.title('Matriz de Confusión')
    plt.show()

## Feature selection:

In [ ]:

#Eliminar columnas con alta correlación, calculada sobre los datos de train.
def drop_correlated_features(X_train, X_test, threshold=0.95): #0.95 threshold be more conservative
    
    # Calculate correlation matrix on TRAIN data
    corr_matrix = X_train.corr().abs()

    # elect the upper triangle to avoid dropping both features in a pair
    upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    # Find features to drop based on the threshold
    to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column] > threshold)]
    
    # Drop
    X_train_filtered = X_train.drop(columns=to_drop)
    X_test_filtered = X_test.drop(columns=to_drop)
    
    return X_train_filtered, X_test_filtered

#APlicar RFE
def apply_rfe(model, X_train, X_test, y_train, num_features=20):
    
    # base estimator
    # class_weight='balanced' ensures it selects features that help with your low stress recall!
    #rf_estimator = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
    
    # Set up RFE
    # step=1 means it drops the single worst feature one at a time for maximum precision
    rfe = RFE(estimator=model, n_features_to_select=num_features, step=1)
    
    # Fit RFE on the training data
    rfe.fit(X_train, y_train)
    
    # Extract the names of the features that survived
    selected_features = X_train.columns[rfe.support_].tolist()
        
    # Apply the selection mask to both Train and Test sets
    X_train_rfe = X_train[selected_features]
    X_test_rfe = X_test[selected_features]
    
    return X_train_rfe, X_test_rfe, selected_features

from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.ensemble import RandomForestClassifier

#Hyperparameter Grid
param_grid = {
    'n_estimators': np.arange(10, 300, 10), # Number of trees in the forest
    'max_depth': [None, 10, 20], # Maximum depth of the tree
    'min_samples_split': [2, 5, 10], # Minimum samples required to split an internal node
    'min_samples_leaf': [1, 2, 4], # Minimum samples required to be at a leaf node
    'max_features': [None, 'sqrt', 'log2'] # Number of features to consider at every split
}


def gridSearch(base_model, param_grid, scoring):
    # Grouped Cross Validation for the Grid Search
    gkf = GroupKFold(n_splits=5)

    # Initialize the Grid Search
    grid_search = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=gkf,
        scoring=scoring, #recall is firstly used as the previous models performed poorly with stressed windows.... BALANCED_ACC OR F1
        n_jobs=-1,
        verbose=2 # Prints progress
    )

    # We need to isolate the subject groups for the training data specifically
    train_idx, test_idx, _, _, subjects = get_group_idx(df)
    grupos_train = subjects.iloc[train_idx] #subjects in train data

    # Fit the Grid Search using only the subjects in train data
    grid_search.fit(X_train, y_train, groups=grupos_train)

    print(f"Best score: {grid_search.best_score_:.4f}")
    print("Best Hyperparameters:")
    for param, value in grid_search.best_params_.items():
        print(f"- {param}: {value}")

    return grid_search.best_estimator_

## 1. Búsqueda de un modelo base mediante GridSearch con tamaño de ventana fijo, sin selección de características.

- Tamaño de ventana = 30 s
- Solapamiento = 10 s

In [ ]:
PATH = '../../StressIDDataset/StressID/StressID Dataset/Physiological'
SR = 500
WINDOW_SIZE = 30
OVERLAP_SEC = 10

data_df = extract_windows(PATH, sr=SR, window_size = WINDOW_SIZE, overlap_sec = OVERLAP_SEC)
features_df = feature_extraction_and_normalization(data_df, threshold=30, by=['Baseline', 'Relax'], clean = True)

base_model = RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE)
param_grid = {
    'n_estimators': np.arange(10, 300, 10), # Number of trees in the forest
    'max_depth': [None, 10, 20], # Maximum depth of the tree
    'min_samples_split': [2, 5, 10], # Minimum samples required to split an internal node
    'min_samples_leaf': [1, 2, 4], # Minimum samples required to be at a leaf node
    'max_features': [None, 'sqrt', 'log2'] # Number of features to consider at every split
}

base_model = gridSearch(base_model = base_model, param_grid = param_grid, scoring = 'f1')

FileNotFoundError: [Errno 2] No such file or directory: '../../StressIDDataset/StressID/StressID Dataset/Physiological'